# Web Data Analysis — Traffic & Engagement Analytics

Analyze acquisition channels, engagement quality, traffic timing, and actionable business opportunities using the supplied web analytics dataset.

**Important:** This project only claims KPIs supported by the dataset. It does not claim revenue, ROI, or conversion performance.

## Business Questions

1. Which acquisition channels generate the most sessions?
2. Which channels have the highest engagement rate?
3. Which channels have the longest engagement time?
4. Which channels generate the most events?
5. Which channels combine high traffic with strong engagement quality?
6. How many sessions are engaged vs non-engaged?
7. How does traffic change over time?
8. Which hours have the highest traffic?
9. Where are engagement opportunities?
10. What practical actions can be recommended from these patterns?

In [ ]:
# 1. Imports
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
print("Libraries imported successfully.")


In [ ]:
# 2. Load the real repository dataset
DATA_PATH = Path("../data/cleaned_web_data.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH.resolve()}\n"
        "Put cleaned_web_data.csv inside the repository's data folder."
    )

df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH.resolve())
print("Raw shape:", df.shape)


In [ ]:
# 3. Inspect raw columns
print(df.columns.tolist())
display(df.head())


In [ ]:
# 4. Clean and standardize the ACTUAL dataset columns
df = df.drop(columns=["Unnamed: 0"], errors="ignore")
df.columns = df.columns.astype(str).str.strip().str.replace("\n", " ", regex=False)

df = df.rename(columns={
    "Session": "Sessions",
    "Engaged Session": "Engaged sessions",
    "Average engagement time per session": "Avg engagement time",
    "Engaged session per user": "Engaged sessions per user"
})

required = [
    "Channel group", "Datehour", "Users", "Sessions",
    "Engaged sessions", "Avg engagement time",
    "Engaged sessions per user", "Events per session",
    "Engagement rate", "Event count"
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns after standardization: {missing}\nAvailable: {df.columns.tolist()}")

numeric = [
    "Users", "Sessions", "Engaged sessions", "Avg engagement time",
    "Engaged sessions per user", "Events per session",
    "Engagement rate", "Event count"
]
for col in numeric:
    df[col] = (df[col].astype(str)
              .str.replace(",", "", regex=False)
              .str.replace("%", "", regex=False)
              .str.strip())
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["Datehour"] = pd.to_datetime(df["Datehour"], errors="coerce")
df["Date"] = df["Datehour"].dt.date
df["Hour"] = df["Datehour"].dt.hour

print("After standardization:", df.shape)
display(df.head())


In [ ]:
# 5. Data-quality diagnostics
quality = pd.DataFrame({
    "Column": df.columns,
    "Missing": df.isna().sum().values,
    "Missing %": (df.isna().mean()*100).round(2).values
})
display(quality)
print("Duplicate rows:", df.duplicated().sum())
print("Date range:", df["Datehour"].min(), "to", df["Datehour"].max())


In [ ]:
# 6. Keep only valid essential rows
before = len(df)
df = df.dropna(subset=["Datehour", "Channel group", "Sessions", "Engaged sessions"]).copy()
df = df[(df["Sessions"] >= 0) & (df["Engaged sessions"] >= 0) & (df["Event count"] >= 0)].copy()

print(f"Rows before validation: {before:,}")
print(f"Rows after validation:  {len(df):,}")
if df.empty:
    raise ValueError("No usable rows remain. Inspect the numeric conversion and CSV values.")
display(df.head())


In [ ]:
# 7. KPI summary
total_sessions = df["Sessions"].sum()
total_engaged = df["Engaged sessions"].sum()
total_events = df["Event count"].sum()

overall_engagement_rate = total_engaged / total_sessions if total_sessions else np.nan
events_per_session = total_events / total_sessions if total_sessions else np.nan
weighted_time = ((df["Avg engagement time"] * df["Sessions"]).sum() / total_sessions
                 if total_sessions else np.nan)

print(f"Total sessions: {total_sessions:,.0f}")
print(f"Engaged sessions: {total_engaged:,.0f}")
print(f"Overall engagement rate: {overall_engagement_rate:.2%}")
print(f"Total events: {total_events:,.0f}")
print(f"Events per session: {events_per_session:.2f}")
print(f"Session-weighted avg engagement time: {weighted_time:.1f} seconds")
print(f"Channels: {df['Channel group'].nunique()}")
print(f"Analysis days: {df['Date'].nunique()}")


In [ ]:
# 8. Channel performance table
channel = (
    df.groupby("Channel group")
      .agg(Sessions=("Sessions","sum"),
           Engaged_Sessions=("Engaged sessions","sum"),
           Events=("Event count","sum"))
      .reset_index()
)

channel["Engagement Rate"] = channel["Engaged_Sessions"] / channel["Sessions"]
channel["Events per Session"] = channel["Events"] / channel["Sessions"]

weighted = (
    df.assign(Weighted_Time=df["Avg engagement time"] * df["Sessions"])
      .groupby("Channel group")
      .agg(Weighted_Time_Sum=("Weighted_Time","sum"),
           Total_Sessions=("Sessions","sum"))
      .reset_index()
)
weighted["Weighted Avg Engagement Time"] = weighted["Weighted_Time_Sum"] / weighted["Total_Sessions"]

channel = channel.merge(
    weighted[["Channel group","Weighted Avg Engagement Time"]],
    on="Channel group", how="left"
)
channel["Non-engaged Sessions"] = channel["Sessions"] - channel["Engaged_Sessions"]
channel["Session Share"] = channel["Sessions"] / channel["Sessions"].sum()
channel = channel.sort_values("Sessions", ascending=False).reset_index(drop=True)

display(channel.style.format({
    "Sessions":"{:,.0f}", "Engaged_Sessions":"{:,.0f}", "Events":"{:,.0f}",
    "Engagement Rate":"{:.2%}", "Events per Session":"{:.2f}",
    "Weighted Avg Engagement Time":"{:.1f}", "Non-engaged Sessions":"{:,.0f}",
    "Session Share":"{:.2%}"
}))


In [ ]:
# 9. Daily and hourly analysis
daily = df.groupby("Date").agg(
    Sessions=("Sessions","sum"),
    Engaged_Sessions=("Engaged sessions","sum"),
    Events=("Event count","sum")
).reset_index()
daily["Engagement Rate"] = daily["Engaged_Sessions"] / daily["Sessions"]
daily = daily.sort_values("Date")

hourly = df.groupby("Hour").agg(
    Sessions=("Sessions","sum"),
    Engaged_Sessions=("Engaged sessions","sum")
).reset_index()
hourly["Engagement Rate"] = hourly["Engaged_Sessions"] / hourly["Sessions"]
hourly = hourly.sort_values("Hour")

peak = hourly.loc[hourly["Sessions"].idxmax()]
print(f"Peak traffic hour: {int(peak['Hour']):02d}:00 with {peak['Sessions']:,.0f} sessions.")


In [ ]:
# 10. Business opportunity view
avg_rate = channel["Engagement Rate"].mean()
opportunities = channel[
    (channel["Sessions"] >= channel["Sessions"].median()) &
    (channel["Engagement Rate"] < avg_rate)
][["Channel group","Sessions","Engagement Rate"]]

print("High-volume channels with below-average engagement:")
display(opportunities)


In [ ]:
# 11. Visualizations
def show_barh(data, value_col, title, xlabel, filename=None):
    p = data.sort_values(value_col, ascending=True)
    plt.figure(figsize=(10,5.5))
    plt.barh(p["Channel group"], p[value_col])
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Channel")
    plt.tight_layout()
    if filename:
        plt.savefig(filename, dpi=200, bbox_inches="tight")
    plt.show()

show_barh(channel, "Sessions", "Sessions by Acquisition Channel", "Sessions")
show_barh(channel, "Engagement Rate", "Engagement Rate by Acquisition Channel", "Engagement Rate")
show_barh(channel, "Weighted Avg Engagement Time", "Session-Weighted Engagement Time by Channel", "Average Engagement Time (seconds)")
show_barh(channel, "Events", "Event Count by Acquisition Channel", "Event Count")

plt.figure(figsize=(12,5.5))
plt.plot(daily["Date"], daily["Sessions"], marker="o", markersize=3)
plt.title("Daily Website Sessions")
plt.xlabel("Date"); plt.ylabel("Sessions")
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

plt.figure(figsize=(11,5.5))
plt.plot(hourly["Hour"], hourly["Sessions"], marker="o")
plt.title("Sessions by Hour of Day")
plt.xlabel("Hour of Day"); plt.ylabel("Sessions")
plt.xticks(range(0,24,2)); plt.tight_layout(); plt.show()

plt.figure(figsize=(10,6))
sizes = np.maximum((channel["Events"]/channel["Events"].max())*1200, 100)
plt.scatter(channel["Sessions"], channel["Engagement Rate"]*100, s=sizes, alpha=0.7)
for _, r in channel.iterrows():
    plt.annotate(r["Channel group"], (r["Sessions"], r["Engagement Rate"]*100),
                 xytext=(5,5), textcoords="offset points")
plt.title("Channel Volume vs Engagement Quality")
plt.xlabel("Sessions"); plt.ylabel("Engagement Rate (%)")
plt.tight_layout(); plt.show()

x = np.arange(len(channel)); width = .38
plt.figure(figsize=(11,5.5))
plt.bar(x-width/2, channel["Engaged_Sessions"], width, label="Engaged Sessions")
plt.bar(x+width/2, channel["Non-engaged Sessions"], width, label="Non-Engaged Sessions")
plt.title("Engaged vs Non-Engaged Sessions by Channel")
plt.xlabel("Channel"); plt.ylabel("Sessions")
plt.xticks(x, channel["Channel group"], rotation=45, ha="right")
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# 12. SAVE THE 8 FINAL PNGs TO THE REPOSITORY /images FOLDER
IMAGES_DIR = Path("../images")
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

# 1 Sessions
p = channel.sort_values("Sessions")
plt.figure(figsize=(10,5.5)); plt.barh(p["Channel group"],p["Sessions"])
plt.title("Sessions by Acquisition Channel"); plt.xlabel("Sessions"); plt.ylabel("Channel")
plt.tight_layout(); plt.savefig(IMAGES_DIR/"01_sessions_by_channel.png",dpi=200,bbox_inches="tight"); plt.close()

# 2 Engagement rate
p = channel.sort_values("Engagement Rate")
plt.figure(figsize=(10,5.5)); plt.barh(p["Channel group"],p["Engagement Rate"]*100)
plt.title("Engagement Rate by Acquisition Channel"); plt.xlabel("Engagement Rate (%)"); plt.ylabel("Channel")
plt.tight_layout(); plt.savefig(IMAGES_DIR/"02_engagement_rate_by_channel.png",dpi=200,bbox_inches="tight"); plt.close()

# 3 Engagement time
p = channel.sort_values("Weighted Avg Engagement Time")
plt.figure(figsize=(10,5.5)); plt.barh(p["Channel group"],p["Weighted Avg Engagement Time"])
plt.title("Session-Weighted Engagement Time by Channel"); plt.xlabel("Average Engagement Time (seconds)"); plt.ylabel("Channel")
plt.tight_layout(); plt.savefig(IMAGES_DIR/"03_engagement_time_by_channel.png",dpi=200,bbox_inches="tight"); plt.close()

# 4 Daily sessions
plt.figure(figsize=(12,5.5)); plt.plot(daily["Date"],daily["Sessions"],marker="o",markersize=3)
plt.title("Daily Website Sessions"); plt.xlabel("Date"); plt.ylabel("Sessions")
plt.xticks(rotation=45); plt.tight_layout(); plt.savefig(IMAGES_DIR/"04_daily_sessions_trend.png",dpi=200,bbox_inches="tight"); plt.close()

# 5 Hourly sessions
plt.figure(figsize=(11,5.5)); plt.plot(hourly["Hour"],hourly["Sessions"],marker="o")
plt.title("Sessions by Hour of Day"); plt.xlabel("Hour of Day"); plt.ylabel("Sessions")
plt.xticks(range(0,24,2)); plt.tight_layout(); plt.savefig(IMAGES_DIR/"05_sessions_by_hour.png",dpi=200,bbox_inches="tight"); plt.close()

# 6 Events
p = channel.sort_values("Events")
plt.figure(figsize=(10,5.5)); plt.barh(p["Channel group"],p["Events"])
plt.title("Event Count by Acquisition Channel"); plt.xlabel("Event Count"); plt.ylabel("Channel")
plt.tight_layout(); plt.savefig(IMAGES_DIR/"06_event_activity_by_channel.png",dpi=200,bbox_inches="tight"); plt.close()

# 7 Volume vs quality
plt.figure(figsize=(10,6))
sizes=np.maximum((channel["Events"]/channel["Events"].max())*1200,100)
plt.scatter(channel["Sessions"],channel["Engagement Rate"]*100,s=sizes,alpha=.7)
for _,r in channel.iterrows():
    plt.annotate(r["Channel group"],(r["Sessions"],r["Engagement Rate"]*100),xytext=(5,5),textcoords="offset points")
plt.title("Channel Volume vs Engagement Quality"); plt.xlabel("Sessions"); plt.ylabel("Engagement Rate (%)")
plt.tight_layout(); plt.savefig(IMAGES_DIR/"07_volume_vs_engagement_quality.png",dpi=200,bbox_inches="tight"); plt.close()

# 8 Engaged vs non-engaged
x=np.arange(len(channel)); width=.38
plt.figure(figsize=(11,5.5))
plt.bar(x-width/2,channel["Engaged_Sessions"],width,label="Engaged Sessions")
plt.bar(x+width/2,channel["Non-engaged Sessions"],width,label="Non-Engaged Sessions")
plt.title("Engaged vs Non-Engaged Sessions by Channel"); plt.xlabel("Channel"); plt.ylabel("Sessions")
plt.xticks(x,channel["Channel group"],rotation=45,ha="right"); plt.legend()
plt.tight_layout(); plt.savefig(IMAGES_DIR/"08_engaged_vs_non_engaged_sessions.png",dpi=200,bbox_inches="tight"); plt.close()

print("✅ Saved 8 PNGs to:", IMAGES_DIR.resolve())
for f in sorted(IMAGES_DIR.glob("*.png")):
    print("✔", f.name)


## Key Insights & Recommendations

Use the calculated tables and charts above to identify:
- highest-volume acquisition channels
- highest-engagement channels
- channels with strong engagement time
- high-volume/low-engagement opportunities
- peak traffic hours

### Recommended business actions
1. Protect and monitor high-volume channels.
2. Study high-engagement channels for reusable content, audience, and landing-page patterns.
3. Investigate high-volume channels with below-average engagement.
4. Align campaigns, content releases, and monitoring with peak traffic periods.
5. Track traffic volume together with engagement quality rather than optimizing for sessions alone.

## Limitations
- No revenue, transaction, acquisition-cost, or customer-lifetime-value fields are present.
- `Users` is not treated as a simple additive unique-user KPI across hourly/channel rows.
- Engagement rate is aggregated as total engaged sessions / total sessions.
- Engagement time is session-weighted to avoid giving equal influence to low- and high-volume rows.
